In [0]:
from pyspark.sql import SparkSession, functions as F

# On Databricks, `spark` is already provided — no need to create a new session
# spark = SparkSession.builder.appName("assignment1").getOrCreate()

DATA = "/Volumes/basicspyspark/default/learn/datasets"
OUT = f"{DATA}/out/assignment1"   # write inside a Unity Catalog volume (DBFS root is disabled on serverless)

ORDERS_SCHEMA = "order_id INT, order_date DATE, customer_id INT, order_status STRING, amount DOUBLE"

orders = (spark.read.option("header", True)
          .schema(ORDERS_SCHEMA)
          .csv(f"{DATA}/orders.csv"))
orders.createOrReplaceTempView("orders")

In [0]:
# TASK-1
# Inferred vs explicit schema (CSV)
orders_inferred = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{DATA}/orders.csv")
)

print("Inferred Schema:")
orders_inferred.printSchema()

print("Explicit Schema:")
orders.printSchema()

Inferred Schema:
root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- amount: double (nullable = true)

Explicit Schema:
root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- amount: double (nullable = true)



In [0]:
# TASK-2
# Read modes on dirty data
dirty_path = f"{DATA}/orders_dirty.csv"

permissive = (
    spark.read
    .schema(ORDERS_SCHEMA)
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .csv(dirty_path)
)

dropmalformed = (
    spark.read
    .schema(ORDERS_SCHEMA)
    .option("header", True)
    .option("mode", "DROPMALFORMED")
    .csv(dirty_path)
)

failfast = (
    spark.read
    .schema(ORDERS_SCHEMA)
    .option("header", True)
    .option("mode", "FAILFAST")
    .csv(dirty_path)
)

print("PERMISSIVE:", permissive.count())
print("DROPMALFORMED:", dropmalformed.count())

try:
    print("FAILFAST:", failfast.count())
except Exception as e:
    print("FAILFAST error:", e)

PERMISSIVE: 45
DROPMALFORMED: 45
FAILFAST: 45


In [0]:
# TASK-3
# Capture the bad rows
from pyspark.sql.types import StructType, StructField, StringType

orders_struct = StructType.fromDDL(ORDERS_SCHEMA)

corrupt_schema = StructType(
    orders_struct.fields +
    [StructField("_corrupt_record", StringType(), True)]
)

dirty_with_errors = (
    spark.read
    .schema(corrupt_schema)
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .csv(dirty_path)
    .cache()
)

dirty_with_errors.count()

dirty_with_errors.filter(
    F.col("_corrupt_record").isNotNull()
).show(truncate=False)

+--------+----------+-----------+------------+-------+-------------------------------+
|order_id|order_date|customer_id|order_status|amount |_corrupt_record                |
+--------+----------+-----------+------------+-------+-------------------------------+
|4       |NULL      |11         |PENDING     |2064.02|4,07/15/2026,11,PENDING,2064.02|
|8       |2026-08-12|27         |ON_HOLD     |NULL   |8,2026-08-12,27,ON_HOLD,abc    |
+--------+----------+-----------+------------+-------+-------------------------------+



In [0]:
# TASK-4
# NESTED JSON
employees = (
    spark.read
    .option("inferSchema", True)
    .json(f"{DATA}/employees.json")
)

employees.printSchema()

employees.select(
    "name",
    F.col("address.city").alias("city"),
    F.col("address.state").alias("state"),
    F.size(F.col("skills")).alias("num_skills")
).show(truncate=False)

root
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- pincode: string (nullable = true)
 |    |-- state: string (nullable = true)
 |-- certifications: struct (nullable = true)
 |    |-- aws: long (nullable = true)
 |    |-- azure: long (nullable = true)
 |    |-- databricks: long (nullable = true)
 |    |-- gcp: long (nullable = true)
 |-- dept: string (nullable = true)
 |-- emp_id: long (nullable = true)
 |-- join_date: string (nullable = true)
 |-- manager_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- skills: array (nullable = true)
 |    |-- element: string (containsNull = true)

+-------------+----------+-----+----------+
|name         |city      |state|num_skills|
+-------------+----------+-----+----------+
|Kavya Kumar  |Kolkata   |WB   |2         |
|Karthik Gupta|Pune      |MH   |4         |
|Ananya Iyer  |Chennai   |TN   |4         |
|Rohan Iyer   |Bengaluru |KA   |3         |
|Ad

In [0]:
# TASK-5
#  Parquet and save modes
parquet_path = f"{OUT}/orders_parquet"

# Initial write
orders.write.mode("overwrite").parquet(parquet_path)

print("Initial rows:", spark.read.parquet(parquet_path).count())

# Default mode: errorifexists
try:
    orders.write.parquet(parquet_path)
except Exception as e:
    print("Default mode error:", str(e)[:300])

print("After default:", spark.read.parquet(parquet_path).count())

# Append
orders.write.mode("append").parquet(parquet_path)
print("After append:", spark.read.parquet(parquet_path).count())

# Ignore
orders.write.mode("ignore").parquet(parquet_path)
print("After ignore:", spark.read.parquet(parquet_path).count())

# Overwrite
orders.write.mode("overwrite").parquet(parquet_path)
print("After overwrite:", spark.read.parquet(parquet_path).count())

Initial rows: 300
Default mode error: [PATH_ALREADY_EXISTS] Path dbfs:/Volumes/basicspyspark/default/learn/datasets/out/assignment1/orders_parquet already exists. Set mode as "overwrite" to overwrite the existing path. SQLSTATE: 42K04

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilatio
After default: 300
After append: 600
After ignore: 600
After overwrite: 300


In [0]:
# TASK-6
# Partionby()
partition_path = f"{OUT}/orders_by_status"

(
    orders.write
    .mode("overwrite")
    .partitionBy("order_status")
    .parquet(partition_path)
)

In [0]:
# TASK-7
customers_path = f"{DATA}/customers.csv"
pipe_path = f"{OUT}/customers_pipe"

customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(customers_path)
)

(
    customers.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .option("sep", "|")
    .csv(pipe_path)
)

In [0]:
# TASK-8
orders.groupBy("order_status") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

orders.agg(
    F.countDistinct("customer_id").alias("active_customers")
).show()

+---------------+-----+
|   order_status|count|
+---------------+-----+
|       COMPLETE|  141|
|         CLOSED|   51|
|        PENDING|   26|
|PENDING_PAYMENT|   25|
|       CANCELED|   16|
|SUSPECTED_FRAUD|   16|
|     PROCESSING|   13|
|        ON_HOLD|   12|
+---------------+-----+

+----------------+
|active_customers|
+----------------+
|              51|
+----------------+



In [0]:
# TASK-9
top5_closed = (
    orders
    .filter(F.col("order_status") == "CLOSED")
    .groupBy("customer_id")
    .count()
    .orderBy(F.desc("count"), F.asc("customer_id"))
    .limit(5)
)

top5_closed.show()

orders.createOrReplaceTempView("orders")
spark.sql("""
SELECT customer_id, COUNT(*) AS count
FROM orders
WHERE order_status = 'CLOSED'
GROUP BY customer_id
ORDER BY count DESC, customer_id ASC
LIMIT 5
""").show()



+-----------+-----+
|customer_id|count|
+-----------+-----+
|          5|    4|
|         16|    3|
|         25|    3|
|         32|    3|
|         46|    3|
+-----------+-----+

+-----------+-----+
|customer_id|count|
+-----------+-----+
|          5|    4|
|         16|    3|
|         25|    3|
|         32|    3|
|         46|    3|
+-----------+-----+



In [0]:

# TASK-10
pending_orders = (
    orders
    .filter(
        F.col("order_status").startswith("PENDING") &
        (
            F.col("amount").isNull() |
            (F.col("amount") < 2000)
        )
    )
    .select("order_id", "order_status", "amount")
)

pending_orders.show()



orders.orderBy(
    F.col("amount").desc_nulls_last()
).limit(10).show()




+--------+---------------+-------+
|order_id|   order_status| amount|
+--------+---------------+-------+
|      15|PENDING_PAYMENT| 1338.6|
|     202|PENDING_PAYMENT|1147.92|
|     228|PENDING_PAYMENT|  892.4|
+--------+---------------+-------+

+--------+----------+-----------+---------------+--------+
|order_id|order_date|customer_id|   order_status|  amount|
+--------+----------+-----------+---------------+--------+
|     140|2026-08-23|          8|         CLOSED|49582.23|
|     132|2026-08-16|         11|        PENDING| 39998.5|
|      80|2026-08-09|          6|        PENDING| 36894.8|
|     164|2026-08-06|         50|SUSPECTED_FRAUD|36008.47|
|      90|2026-08-07|         17|       COMPLETE| 34371.4|
|     150|2026-07-11|         28|       COMPLETE|34360.25|
|     271|2026-07-30|         24|         CLOSED|33409.65|
|     106|2026-08-25|         23|SUSPECTED_FRAUD|32716.45|
|     247|2026-07-31|         39|       COMPLETE|32373.12|
|      47|2026-07-18|          6|       COMPLE

In [0]:
# TASK-11
orders.agg(
    F.count("*").alias("total_orders"),
    F.count("amount").alias("orders_with_amount"),
    F.sum("amount").alias("total_revenue"),
    F.round(F.avg("amount"), 2).alias("average_amount"),
    F.min("amount").alias("smallest_amount"),
    F.max("amount").alias("largest_amount")
).show(truncate=False)

+------------+------------------+------------------+--------------+---------------+--------------+
|total_orders|orders_with_amount|total_revenue     |average_amount|smallest_amount|largest_amount|
+------------+------------------+------------------+--------------+---------------+--------------+
|300         |290               |4022081.4399999976|13869.25      |446.2          |49582.23      |
+------------+------------------+------------------+--------------+---------------+--------------+



In [0]:
# TASK-12
daily_revenue = (
    orders
    .groupBy("order_date")
    .agg(
        F.count("*").alias("order_count"),
        F.sum("amount").alias("total_revenue"),
        F.round(F.avg("amount"), 2).alias("average_amount")
    )
    .filter(F.col("total_revenue") > 100000)
    .orderBy(F.desc("total_revenue"))
)

daily_revenue.show(truncate=False)

+----------+-----------+------------------+--------------+
|order_date|order_count|total_revenue     |average_amount|
+----------+-----------+------------------+--------------+
|2026-07-27|8          |143761.19         |17970.15      |
|2026-07-26|11         |138291.4          |13829.14      |
|2026-08-24|9          |130705.93         |14522.88      |
|2026-07-29|6          |127772.67         |21295.45      |
|2026-07-09|9          |126313.09         |14034.79      |
|2026-08-07|8          |122984.95999999999|20497.49      |
|2026-08-05|6          |113451.45000000001|18908.58      |
|2026-08-22|8          |110104.37999999999|13763.05      |
+----------+-----------+------------------+--------------+

